# UdaPlay - Part 01: Offline RAG

Build a persistent ChromaDB vector store over the local `games/` JSON dataset and demonstrate semantic search.

In [ ]:
import importlib.util, sys
if importlib.util.find_spec('pysqlite3') is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [ ]:
import os, json, glob
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

load_dotenv()
assert os.getenv('OPENAI_API_KEY'), 'Missing OPENAI_API_KEY in .env'
os.environ.setdefault('CHROMA_OPENAI_API_KEY', os.environ['OPENAI_API_KEY'])

In [ ]:
CHROMA_PATH = 'chromadb'
COLLECTION_NAME = 'udaplay'

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.environ['CHROMA_OPENAI_API_KEY'],
    model_name='text-embedding-3-small',
)
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_fn,
    metadata={'hnsw:space': 'cosine'},
)
print('Existing docs in collection:', collection.count())

In [ ]:
DATA_DIR = 'games'
ids, docs, metas = [], [], []
for path in sorted(glob.glob(os.path.join(DATA_DIR, '*.json'))):
    with open(path, 'r', encoding='utf-8') as f:
        game = json.load(f)
    doc_id = os.path.splitext(os.path.basename(path))[0]
    content = (
        f"Name: {game['Name']}\n"
        f"Platform: {game['Platform']}\n"
        f"Genre: {game['Genre']}\n"
        f"Publisher: {game['Publisher']}\n"
        f"Year: {game['YearOfRelease']}\n"
        f"Description: {game['Description']}"
    )
    ids.append(doc_id)
    docs.append(content)
    metas.append(game)

collection.upsert(ids=ids, documents=docs, metadatas=metas)
print(f'Indexed {len(ids)} games. Collection size = {collection.count()}')

In [ ]:
def search(query: str, k: int = 3):
    res = collection.query(query_texts=[query], n_results=k)
    out = []
    for i in range(len(res['ids'][0])):
        out.append({
            'id': res['ids'][0][i],
            'distance': res['distances'][0][i],
            'metadata': res['metadatas'][0][i],
            'document': res['documents'][0][i],
        })
    return out

for q in [
    'first 3D Mario platformer',
    'open world Rockstar game on PS2',
    'racing simulator on PlayStation',
]:
    print('Q:', q)
    for hit in search(q, 2):
        m = hit['metadata']
        print(f"  - {m['Name']} ({m['Platform']}, {m['YearOfRelease']})  dist={hit['distance']:.3f}")
    print()

The persistent collection at `./chromadb` is now ready to be reused by Part 2 (the agent).